# 05 · Training

Runs a smoke version of the three-phase trainer (small dataset, few
epochs) so the notebook executes in minutes on CPU. For the full run
use the CLI:

```bash
python train.py --config configs/default.yaml
```

The trainer writes a per-run history JSON to `checkpoints/runs/<ts>/`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data.augmentations import build_eval_transforms, build_train_transforms
from src.data.catalog import load_hyg_catalog
from src.data.dataset import SyntheticStarFieldDataset
from src.models.astrolocnet import AstroLocNet
from src.training.trainer import PhaseConfig, Trainer, TrainerConfig

In [ ]:
catalog = load_hyg_catalog(ROOT / 'data/catalogs/hygdata_v3.csv', mag_limit=8.0)
train_ds = SyntheticStarFieldDataset(catalog, n_samples=128, transform=build_train_transforms(224), seed=1)
val_ds   = SyntheticStarFieldDataset(catalog, n_samples=64,  transform=build_eval_transforms(224),  seed=99)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False)

In [ ]:
events = []
def cb(evt): events.append(evt); print({k: evt[k] for k in ('phase','epoch','val') if k in evt})

model = AstroLocNet(pretrained=True)
cfg = TrainerConfig(
    batch_size=16, amp=False, device='cpu', early_stopping_patience=5,
    phase1=PhaseConfig(epochs=1, lr_head=1e-3),
    phase2=PhaseConfig(epochs=1, lr_head=1e-4, lr_backbone_late=5e-5, unfreeze_last_n_blocks=2),
)
trainer = Trainer(model, cfg, progress_cb=cb)
trainer.run_phase1(train_loader, val_loader)
trainer.run_phase2(train_loader, val_loader)
print('Best val angular separation:', trainer.best_metric)

## Training curves

In [ ]:
import pandas as pd
rows = []
for h in trainer.history:
    rows.append({'phase': h['phase'], 'epoch': h['epoch'],
                 'train_loss': h['train']['loss'], 'val_loss': h['val']['loss'],
                 'val_sep_deg': h['val']['ang_sep_mean_deg']})
df = pd.DataFrame(rows); df['x'] = range(len(df))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(df['x'], df['train_loss'], label='train', color='#7c5cff')
axes[0].plot(df['x'], df['val_loss'], label='val', color='#ffd166')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('epoch (across phases)')
axes[1].plot(df['x'], df['val_sep_deg'], color='#3ad29f', marker='o')
axes[1].set_title('Validation angular separation (°)'); axes[1].set_xlabel('epoch (across phases)')
for i in range(1, len(df)):
    if df['phase'].iloc[i] != df['phase'].iloc[i - 1]:
        for a in axes: a.axvline(i - 0.5, color='gray', linestyle='--', alpha=0.4)
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/05_training_curves.png', dpi=150)
plt.show()